# Cruces entre datasets (caso 1.3)

El caso 1.3 pide **agrupar los distritos de Madrid según su perfil de riesgo y su contexto social**, cruzando el socioeconómico con las activaciones (por ejemplo, ver si la renta media se relaciona con ciertos tipos de emergencia, o si la densidad influye en los tiempos de llegada). Bluetab nos pidió además mirar qué otros datasets se pueden cruzar para sacar más información.

Aquí revisamos dataset a dataset qué cruces son posibles, cuáles usamos y por qué, y montamos la tabla final: **una fila por distrito**, que es la que entra en el clustering.

**Orden:** se ejecuta después de `limpieza_socioeconomico.ipynb`, `limpieza_distritos.ipynb` y `limpieza_activaciones.ipynb`, porque usa los CSV limpios que generan.

In [1]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

def cargar(nombre):
    # Los CSV limpios están en ../data_clean; los catálogos originales en ../data.
    ruta = Path(nombre)
    if not ruta.exists():
        ruta = Path('../data_clean') / nombre
    if not ruta.exists():
        ruta = Path('../data') / nombre
    return pd.read_csv(ruta)

def normalizar_texto(valor):
    if pd.isna(valor):
        return None
    s = unicodedata.normalize('NFD', str(valor).strip().upper())
    return ''.join(c for c in s if unicodedata.category(c) != 'Mn')

soc = cargar('socioeconomico_limpio.csv')
dis = cargar('distritos_limpio.csv')
act = cargar('activaciones_limpio.csv')

print(f"socioeconomico: {soc.shape}, distritos: {dis.shape}, activaciones: {act.shape}")

socioeconomico: (21, 15), distritos: (21, 7), activaciones: (152107, 13)


## 1. Socioeconómico + distritos: la base

Las dos tablas tienen una fila por distrito y el mismo `distrito_id`, así que se unen directo y encajan las 21 de 21. Es la tabla maestra: de aquí salen población, superficie y densidad, coherentes con los indicadores sociales.

In [2]:
# Quitamos del socioeconomico las columnas que ya estan en distritos (poblacion y superficie),
# porque son las mismas (se comprobo en la limpieza) y si no saldrian duplicadas.
base = dis.merge(soc.drop(columns=['poblacion_2024', 'superficie_km2']), on='distrito_id')
print(f"Filas tras unir: {len(base)} (de 21)")
print(f"Distritos sin pareja: {base['distrito'].isna().sum()}")
base[['distrito_id', 'nombre', 'poblacion', 'superficie_km2', 'renta_media_hogar',
      'indice_vulnerabilidad', 'densidad_hab_km2']].head()

Filas tras unir: 21 (de 21)
Distritos sin pareja: 0


,distrito_id,nombre,poblacion,superficie_km2,renta_media_hogar,indice_vulnerabilidad,densidad_hab_km2
0,1,CENTRO,149718,5.23,42831.0,0.35,28626.8
1,2,ARGANZUELA,155115,6.47,39876.0,0.42,23974.5
2,3,RETIRO,119915,5.46,61604.0,0.22,21962.5
3,4,SALAMANCA,147707,5.41,71432.0,0.15,27302.6
4,5,CHAMARTIN,146512,9.19,62145.0,0.18,15942.5


## 2. Activaciones por distrito: la demanda

Las activaciones ya traen el distrito normalizado de la limpieza. Quedan fuera del análisis por distrito las 876 filas con distrito `NO CONSTA` (0,6%) y 201 filas de fuera del término municipal (`FUERA DE TERMINO` y `CARRETERAS Y CIRCUNVALACIONES`). Siguen en el CSV limpio y cuentan en la demanda total de Madrid, pero no pertenecen a ningún distrito de la ciudad.

Agrupamos por distrito y calculamos:

- **Nº de activaciones** y **activaciones por 1.000 habitantes**. Dividimos por población porque si no, los distritos grandes siempre parecen peores solo por tener más gente.
- **% de prioridad alta**, como indicador de gravedad de la demanda.
- **Edad media de los pacientes**, que se puede comparar con la edad media del distrito.

- **Tiempo medio de respuesta**, calculado en la limpieza de activaciones con la regla de medianoche que pidió Bluetab (mismo día salvo cruce de 20:00-23:59 a 00:00-06:59, que suma 24 h; las incoherencias quedan vacías sin borrar la activación).

In [3]:
# Solo los 21 distritos de Madrid ciudad: fuera NO CONSTA y las activaciones de fuera del termino
act_distrito = act[act['distrito'].isin(set(dis['nombre']))].copy()
print(f"Activaciones con distrito asignado: {len(act_distrito)} de {len(act)} "
      f"(quedan fuera {len(act) - len(act_distrito)}, el "
      f"{(len(act) - len(act_distrito)) / len(act) * 100:.2f}%)")

demanda = act_distrito.groupby('distrito').agg(
    n_activaciones=('activacion_id', 'count'),
    pct_prioridad_alta=('prioridad', lambda s: (s == 'ALTA').mean() * 100),
    edad_media_paciente=('paciente_edad', 'mean'),
    tiempo_respuesta_medio_min=('tiempo_respuesta_min', 'mean'),
).round(2).reset_index()

print(f"Distritos con activaciones: {len(demanda)}")
demanda.head()

Activaciones con distrito asignado: 151030 de 152107 (quedan fuera 1077, el 0.71%)
Distritos con activaciones: 21


,distrito,n_activaciones,pct_prioridad_alta,edad_media_paciente,tiempo_respuesta_medio_min
0,ARGANZUELA,8104,29.73,49.29,7.27
1,BARAJAS,2385,30.36,50.65,11.98
2,CARABANCHEL,10642,29.73,49.40,8.17
3,CENTRO,22144,29.79,50.03,7.92
4,CHAMARTIN,7260,30.29,48.98,9.37


## 3. Qué aportan los demás datasets

Bluetab nos dijo que mirásemos otros datasets para cruzar información. Esto es lo que hemos encontrado, dataset a dataset.

In [4]:
uni = cargar('unidades_samur_limpio.csv')
bases = cargar('bases_samur_limpio.csv')
hosp = cargar('hospitales_limpio.csv')
acc = cargar('accidentes.csv')

# UNIDADES SAMUR (catalogo limpio): encaja por unidad_id -> carga de trabajo por unidad y por base.
match_uni = act['unidad_id'].isin(set(uni['unidad_id'])).mean() * 100
print(f"1. unidades_samur: encaja el {match_uni:.0f}% por unidad_id (catalogo limpio, {len(uni)} unidades).")

# TIPOS DE EMERGENCIA: el problema del catalogo original ya esta resuelto (caso 2.2 + analisis).
tipos_raw = cargar('tipos_emergencia.csv')
tipos_clean = cargar('tipos_emergencia_clean.csv')
tipos_raw['cod_norm'] = tipos_raw['codigo'].map(normalizar_texto)
dups = tipos_raw[tipos_raw.duplicated('cod_norm', keep=False)].sort_values('cod_norm')
afectadas = act['tipo_emergencia'].map(normalizar_texto).isin(set(dups['cod_norm'])).sum()
match_clean = act['tipo_emergencia'].map(normalizar_texto).isin(set(tipos_clean['codigo'].map(normalizar_texto)))
print(f"2. tipos_emergencia: el catalogo original traia {len(dups)//2} codigos repetidos con categoria "
      f"y prioridad distintas ({afectadas} activaciones afectadas). Ese diagnostico es historico: "
      f"el caso 2.2 dejo tipos_emergencia_clean.csv ({len(tipos_clean)} tipos) con las fusiones "
      f"confirmadas por Bluetab (correo 14/10) y las correcciones en columnas nuevas.")
print(f"   Match directo por texto normalizado con el catalogo limpio: {match_clean.mean()*100:.1f}% "
      f"({int(match_clean.sum())} de {len(act)}).")
print("   El cruce canonico completo (con las equivalencias de fusion) esta hecho en el notebook de "
      "analisis (seccion 4): 142.150 clasificadas de 152.107 (93,45%) y 9.957 fuera de catalogo.")

# BASES Y HOSPITALES (catalogos limpios): catalogos por distrito -> oferta sanitaria.
print(f"3. bases_samur: {len(bases)} bases; hospitales: {len(hosp)} hospitales (catalogos limpios). "
      f"Se cruzan por distrito (normalizando el nombre).")

# ACCIDENTES: trafico, con municipios de fuera de Madrid y distritos que faltan.
en_catalogo = acc['distrito'].map(normalizar_texto).isin(set(dis['nombre'])).sum()
print(f"4. accidentes: solo {en_catalogo} de {len(acc)} filas tienen un distrito de Madrid ciudad; "
      f"el resto son municipios de fuera o nombres ambiguos.")

# METEOROLOGIA Y CALENDARIO: van por dia y activaciones no tiene dia.
print("5. meteorologia y calendario: van por dia. Activaciones solo tiene año, mes y hora, "
      "asi que no se pueden cruzar por fecha.")

1. unidades_samur: encaja el 100% por unidad_id (catalogo limpio, 60 unidades).


2. tipos_emergencia: el catalogo original traia 2 codigos repetidos con categoria y prioridad distintas (14223 activaciones afectadas). Ese diagnostico es historico: el caso 2.2 dejo tipos_emergencia_clean.csv (42 tipos) con las fusiones confirmadas por Bluetab (correo 14/10) y las correcciones en columnas nuevas.
   Match directo por texto normalizado con el catalogo limpio: 93.4% (142115 de 152107).
   El cruce canonico completo (con las equivalencias de fusion) esta hecho en el notebook de analisis (seccion 4): 142.150 clasificadas de 152.107 (93,45%) y 9.957 fuera de catalogo.
3. bases_samur: 15 bases; hospitales: 20 hospitales (catalogos limpios). Se cruzan por distrito (normalizando el nombre).
4. accidentes: solo 1596 de 2062 filas tienen un distrito de Madrid ciudad; el resto son municipios de fuera o nombres ambiguos.
5. meteorologia y calendario: van por dia. Activaciones solo tiene año, mes y hora, asi que no se pueden cruzar por fecha.


### Lo que decidimos con cada uno

- **unidades_samur: sí.** Encaja el 100% por `unidad_id` (unir por id, no por el código, que se repite). Lo usamos para la oferta: cuántas unidades tiene cada distrito en su base. Ojo: con el catálogo limpio hay 10 unidades cuya base pone `VALLECAS` a secas y 5 con base `NO CONSTA` o vacía. Esas 15 no se reparten entre distritos, porque no hay forma de saber a cuál pertenecen (ver sección 4).
- **bases_samur y hospitales: sí.** Se cruzan por distrito y dan la oferta sanitaria: bases, unidades, hospitales y camas. El caso pregunta justo por la relación entre densidad y recursos del SAMUR. Los catálogos limpios ya traen los nombres de distrito casi listos, y aplicamos la misma regla de normalización de siempre (ver más abajo). Además comparamos sus conteos con las columnas `num_hospitales` y `base_samur` de `distritos.csv`, y no coinciden en varios distritos (ver la comprobación de abajo). Y un cambio importante respecto a la primera versión: las camas que no constan en el catálogo se quedan **vacías, sin imputar** (antes se les ponía la media, y eso era inventar un dato).
- **tipos_emergencia: ya resuelto, fuera de este notebook.** El diagnóstico del catálogo original (2 códigos repetidos con categoría y prioridad distintas) es histórico: llevó a crear `tipos_emergencia_clean.csv` en el caso 2.2, con las fusiones confirmadas por Bluetab (correo 14/10) y las correcciones documentadas en columnas nuevas sin tocar el original. El cruce canónico de las activaciones con ese catálogo está hecho en el notebook de análisis (sección 4): 142.150 de 152.107 clasificadas (93,45%) y 9.957 fuera de catálogo, sin inventar equivalencias. Para la tabla por distrito no hace falta cruzarlo: el mix de tipos por distrito sigue previsto como posible ampliación.
- **accidentes: no para el clustering.** Es de tráfico, incluye municipios de fuera de Madrid (Alcorcón, Getafe, Móstoles...) que no están en el socioeconómico, y le faltan distritos. Se queda como posible ampliación (riesgo vial por distrito).
- **meteorología y calendario: no se puede.** Van por día y las activaciones no tienen día: el dato abierto del SAMUR solo publica año, mes y horas (lo confirmó Bluetab por correo el 14/10), así que es una limitación del dato. Cruzar solo por año-mes sería una asociación demasiado gruesa para el clustering.
- **Vallecas a secas (39 activaciones):** Bluetab sugirió mirar si otros datasets lo resolvían. Probamos unidades_samur y hospitales y no distinguen entre Puente y Villa de Vallecas; accidentes, meteorología y calendario no comparten clave con esas filas. El detalle está en `limpieza_activaciones.ipynb` (sección "Vallecas a secas") y la decisión es mantenerlas como NO CONSTA.

## 4. Tabla final: una fila por distrito

Juntamos la base (socioeconómico + distritos) con la demanda (activaciones agrupadas) y la oferta (bases, unidades, hospitales y camas). El resultado es `tabla_distritos.csv`: 21 filas, una por distrito, con el perfil social, la demanda y los recursos. Es la tabla que entra en el clustering.

Los catálogos limpios ya traen el distrito en mayúsculas y sin tildes, y casi todos encajan exactos con los nombres oficiales. Aun así aplicamos la misma regla de siempre (no una lista escrita a mano): se quita el artículo del principio (`LA LATINA` → `LATINA`) y, si el texto es el principio de un único nombre oficial, se completa (`FUENCARRAL` → `FUENCARRAL-EL PARDO`). Lo que no encaja de forma única **no se asigna**:

- **10 unidades con base `VALLECAS`:** puede ser Puente o Villa de Vallecas y no hay forma de distinguirlo, así que cuentan en el total de Madrid pero no en ningún distrito.
- **5 unidades con base `NO CONSTA` o vacía:** tampoco se reparten.
- **9 hospitales de otros municipios** (Getafe, Móstoles, Alcorcón...): son reales pero fuera de Madrid ciudad; quedan fuera de la tabla por distrito.
- **El Hospital Infanta Leonor** figura con distrito `VALLECAS`: mismo problema, no se asigna a Puente ni a Villa.

Consecuencia: **Puente de Vallecas y Villa de Vallecas no tienen un cero seguro en unidades, hospitales ni camas**, porque hay recursos del catálogo que pueden ser de cualquiera de los dos. En esas celdas la tabla deja vacío (desconocido), no 0. Y **Arganzuela** tiene un hospital cuyas camas no constan en el catálogo, así que sus camas también quedan vacías: la suma de camas conocidas no es la capacidad real si falta el dato de un hospital. Los ceros que sí quedan son ceros reales según el catálogo: distritos sin ningún registro de ese recurso.

Mientras esas celdas sigan vacías, las columnas `unidades_samur`, `num_hospitales_cat` y `camas_hospital` **no están listas para el clustering**: falta decidir cómo tratar el desconocido (por ejemplo, preguntando a Bluetab) antes de usarlas. El resto de la tabla está completa.

In [5]:
# Oferta por distrito: bases, unidades y hospitales (catalogos limpios)
oficiales = set(dis['nombre'])

def completar_distrito(texto, candidatos, contiene=False):
    t = normalizar_texto(texto)
    if t is None:
        return None
    for articulo in ('LA ', 'EL ', 'LOS ', 'LAS '):
        if t.startswith(articulo):
            t = t[len(articulo):]
    if t in candidatos:
        return t
    # principio de un nombre (o texto contenido) y solo si es uno
    posibles = [c for c in candidatos if (t in c if contiene else c.startswith(t))]
    return posibles[0] if len(posibles) == 1 else None

bases['distrito_norm'] = bases['distrito'].apply(lambda x: completar_distrito(x, oficiales))
hosp['distrito_norm'] = hosp['distrito'].apply(lambda x: completar_distrito(x, oficiales))
uni['distrito_norm'] = uni['base_distrito'].apply(lambda x: completar_distrito(x, oficiales))

print("Bases sin distrito oficial:", bases.loc[bases['distrito_norm'].isna(), 'distrito'].tolist())
print("Hospitales sin distrito de Madrid:", hosp.loc[hosp['distrito_norm'].isna(), 'distrito'].tolist())
print("Unidades sin distrito asignable:", int(uni['distrito_norm'].isna().sum()), "de", len(uni),
      "(10 con base VALLECAS, que no distingue Puente/Villa, y 5 con base NO CONSTA o vacia)")
print()

oferta_bases = bases.groupby('distrito_norm').agg(num_bases_samur=('base_id', 'count')).reset_index()
oferta_unidades = uni.groupby('distrito_norm').agg(unidades_samur=('unidad_id', 'count')).reset_index()
oferta_hosp = hosp.groupby('distrito_norm').agg(
    num_hospitales_cat=('hospital_id', 'count'), camas_hospital=('camas', lambda s: s.sum(min_count=1))).reset_index()

tabla = (base
         .merge(demanda, left_on='nombre', right_on='distrito', how='left')
         .merge(oferta_bases, left_on='nombre', right_on='distrito_norm', how='left')
         .merge(oferta_unidades, left_on='nombre', right_on='distrito_norm', how='left')
         .merge(oferta_hosp, left_on='nombre', right_on='distrito_norm', how='left'))

tabla['activaciones_por_1000hab'] = (tabla['n_activaciones'] / tabla['poblacion'] * 1000).round(2)

# Ceros reales: un distrito que no aparece en un catalogo no tiene ese recurso segun el catalogo.
# Excepcion: un hospital que si sale pero sin dato de camas no es 0 camas, es desconocido (vacio).
tabla['num_bases_samur'] = tabla['num_bases_samur'].fillna(0).astype(int)
tabla['unidades_samur'] = tabla['unidades_samur'].fillna(0)
tabla['num_hospitales_cat'] = tabla['num_hospitales_cat'].fillna(0)
tabla.loc[tabla['num_hospitales_cat'] == 0, 'camas_hospital'] = tabla.loc[tabla['num_hospitales_cat'] == 0, 'camas_hospital'].fillna(0)

# ...pero hay recursos que pueden ser de Puente o de Villa de Vallecas sin que sepamos de cual
# (10 unidades con base VALLECAS y el Hospital Infanta Leonor con distrito VALLECAS).
# Para esos dos distritos el 0 seria falso: se deja vacio (desconocido), no cero.
VALLECAS_DUDOSOS = ['PUENTE DE VALLECAS', 'VILLA DE VALLECAS']
for c in ['unidades_samur', 'num_hospitales_cat', 'camas_hospital']:
    tabla.loc[tabla['nombre'].isin(VALLECAS_DUDOSOS), c] = np.nan

print("Celdas vacias en la oferta (desconocido, no cero):")
for c in ['unidades_samur', 'num_hospitales_cat', 'camas_hospital']:
    falta = tabla.loc[tabla[c].isna(), 'nombre'].tolist()
    print(f"  {c}: {falta}")

cols_tabla = ['distrito_id', 'nombre', 'poblacion', 'superficie_km2', 'densidad_hab_km2',
              'renta_media_hogar', 'edad_media', 'pct_mayores_65', 'pct_extranjeros',
              'tasa_paro', 'indice_vulnerabilidad', 'num_centros_salud', 'parques_m2_habitante',
              'n_activaciones', 'activaciones_por_1000hab', 'pct_prioridad_alta',
              'edad_media_paciente', 'tiempo_respuesta_medio_min', 'num_bases_samur', 'unidades_samur',
              'num_hospitales_cat', 'camas_hospital']
tabla = tabla[cols_tabla].rename(columns={'nombre': 'distrito'})
for c in ['unidades_samur', 'num_hospitales_cat', 'camas_hospital']:
    tabla[c] = tabla[c].astype('Int64')   # enteros con vacio permitido
tabla.to_csv('tabla_distritos.csv', index=False)
print(tabla.shape)   # (21, 22)
tabla.head(10)

Bases sin distrito oficial: []
Hospitales sin distrito de Madrid: ['GETAFE', 'LEGANES', 'ALCALA DE HENARES', 'MOSTOLES', 'VALLECAS', 'COSLADA', 'SAN SEBASTIAN DE LOS REYES', 'MOSTOLES', 'PARLA', 'ALCORCON']
Unidades sin distrito asignable: 15 de 60 (10 con base VALLECAS, que no distingue Puente/Villa, y 5 con base NO CONSTA o vacia)

Celdas vacias en la oferta (desconocido, no cero):
  unidades_samur: ['PUENTE DE VALLECAS', 'VILLA DE VALLECAS']
  num_hospitales_cat: ['PUENTE DE VALLECAS', 'VILLA DE VALLECAS']
  camas_hospital: ['ARGANZUELA', 'PUENTE DE VALLECAS', 'VILLA DE VALLECAS']
(21, 22)


,distrito_id,distrito,poblacion,superficie_km2,densidad_hab_km2,renta_media_hogar,edad_media,pct_mayores_65,pct_extranjeros,tasa_paro,...,parques_m2_habitante,n_activaciones,activaciones_por_1000hab,pct_prioridad_alta,edad_media_paciente,tiempo_respuesta_medio_min,num_bases_samur,unidades_samur,num_hospitales_cat,camas_hospital
0,1,CENTRO,149718,5.23,28626.8,42831.0,44.2,22.1,21.5,8.2,...,3.2,22144,147.90,29.79,50.03,7.92,1,3,0,0
1,2,ARGANZUELA,155115,6.47,23974.5,39876.0,42.8,18.5,18.2,9.5,...,8.5,8104,52.25,29.73,49.29,7.27,1,0,1,<NA>
2,3,RETIRO,119915,5.46,21962.5,61604.0,47.3,25.3,12.3,6.8,...,25.6,5380,44.87,29.29,49.12,7.09,0,0,1,1500
3,4,SALAMANCA,147707,5.41,27302.6,71432.0,48.1,26.8,11.5,5.2,...,4.8,7705,52.16,29.84,49.31,6.82,0,0,1,480
4,5,CHAMARTIN,146512,9.19,15942.5,62145.0,46.5,25.2,10.8,5.8,...,6.2,7260,49.55,30.29,48.98,9.37,1,0,1,150
5,6,TETUAN,155159,5.37,28893.7,35678.0,41.2,17.8,22.4,11.2,...,2.8,8106,52.24,30.25,49.86,7.09,1,11,0,0
6,7,CHAMBERI,137532,4.68,29387.2,51234.0,45.8,23.5,13.5,7.5,...,3.5,7123,51.79,29.33,49.46,6.68,0,0,0,0
7,8,FUENCARRAL-EL PARDO,245974,237.84,1034.2,48765.0,42.1,16.2,12.1,8.1,...,45.2,6302,25.62,29.69,49.54,9.85,1,6,1,1328
8,9,MONCLOA-ARAVACA,123606,46.29,2670.3,55432.0,43.5,19.8,11.2,7.6,...,18.5,8422,68.14,30.04,49.04,8.23,1,0,2,1574
9,10,LATINA,233765,25.37,9214.2,31245.0,43.8,20.5,18.5,12.5,...,5.8,8625,36.90,30.02,49.53,8.60,1,11,0,0


### Comprobación: `distritos.csv` frente a los catálogos

En `distritos.csv` vienen `num_hospitales` y `base_samur` por distrito. Los comparamos con lo que sale de contar los catálogos limpios de hospitales y bases. No corregimos ninguna de las dos fuentes: solo vemos dónde no coinciden.

In [6]:
comparacion = dis[['nombre', 'num_hospitales', 'base_samur']].merge(
    oferta_hosp.rename(columns={'distrito_norm': 'nombre'})[['nombre', 'num_hospitales_cat']], on='nombre', how='left').merge(
    oferta_bases.rename(columns={'distrito_norm': 'nombre'}), on='nombre', how='left')
comparacion[['num_hospitales_cat', 'num_bases_samur']] = comparacion[['num_hospitales_cat', 'num_bases_samur']].fillna(0)
comparacion['base_catalogo'] = (comparacion['num_bases_samur'] > 0).astype(int)

dif_hosp = comparacion[comparacion['num_hospitales'] != comparacion['num_hospitales_cat']]
dif_base = comparacion[comparacion['base_samur'] != comparacion['base_catalogo']]
print(f"Hospitales: no coinciden en {len(dif_hosp)} de 21 distritos")
print(dif_hosp[['nombre', 'num_hospitales', 'num_hospitales_cat']].to_string(index=False))
print()
print(f"Base SAMUR: no coincide en {len(dif_base)} de 21 distritos")
print(dif_base[['nombre', 'base_samur', 'base_catalogo']].to_string(index=False))

Hospitales: no coinciden en 12 de 21 distritos
             nombre  num_hospitales  num_hospitales_cat
             CENTRO               3                 0.0
         ARGANZUELA               2                 1.0
          SALAMANCA               2                 1.0
          CHAMARTIN               3                 1.0
             TETUAN               1                 0.0
           CHAMBERI               2                 0.0
FUENCARRAL-EL PARDO               2                 1.0
             LATINA               1                 0.0
 PUENTE DE VALLECAS               1                 0.0
      CIUDAD LINEAL               2                 0.0
         VILLAVERDE               1                 0.0
SAN BLAS-CANILLEJAS               1                 0.0

Base SAMUR: no coincide en 6 de 21 distritos
         nombre  base_samur  base_catalogo
         TETUAN           0              1
       CHAMBERI           1              0
MONCLOA-ARAVACA           0              1
       

## 5. Primer vistazo: renta, vulnerabilidad y demanda

El caso pregunta si los barrios más vulnerables tienen una demanda distinta. Como primer vistazo (sin sacar conclusiones todavía, que eso es el clustering), miramos las correlaciones entre la demanda por 1.000 habitantes y los indicadores sociales, y los tipos de emergencia más frecuentes.

In [7]:
cols_corr = ['activaciones_por_1000hab', 'renta_media_hogar', 'indice_vulnerabilidad',
             'pct_mayores_65', 'tasa_paro', 'densidad_hab_km2', 'edad_media_paciente']
print("Correlaciones con la demanda por 1.000 habitantes:")
print(tabla[cols_corr].corr()['activaciones_por_1000hab'].round(2).sort_values())
print()
print("Tipos de emergencia mas frecuentes (top 10):")
print(act['tipo_emergencia'].value_counts().head(10).to_string())
print()
tabla.describe().round(2)

Correlaciones con la demanda por 1.000 habitantes:
indice_vulnerabilidad      -0.27
tasa_paro                  -0.27
renta_media_hogar           0.19
edad_media_paciente         0.21
pct_mayores_65              0.31
densidad_hab_km2            0.44
activaciones_por_1000hab    1.00
Name: activaciones_por_1000hab, dtype: float64

Tipos de emergencia mas frecuentes (top 10):
tipo_emergencia
PATOLOGIA CARDIOVASCULAR         28144
CASUAL: CAIDA, ETC               26013
INTOXICACION ETILICA             14036
AGRESION SIN ESPECIFICAR          8611
ACCIDENTE MENOS DE 3 VICTIMAS     7877
HERIDAS                           6690
PATOLOGIA PSICOSOMATICA           5732
CERTIFICACION PSIQUIATRICO        5359
OTROS                             5188
ACCIDENTE DE MOTOCICLETA          4675



,distrito_id,poblacion,superficie_km2,densidad_hab_km2,renta_media_hogar,edad_media,pct_mayores_65,pct_extranjeros,tasa_paro,indice_vulnerabilidad,...,parques_m2_habitante,n_activaciones,activaciones_por_1000hab,pct_prioridad_alta,edad_media_paciente,tiempo_respuesta_medio_min,num_bases_samur,unidades_samur,num_hospitales_cat,camas_hospital
count,21.0,21.00,21.00,21.00,21.00,21.00,21.00,21.00,21.00,21.00,...,21.00,21.00,21.00,21.00,21.00,21.00,21.00,19.0,19.0,18.0
mean,11.0,157604.19,28.80,14410.11,41709.81,42.95,18.95,16.55,10.54,0.46,...,11.97,7191.90,47.04,29.67,49.48,8.57,0.71,2.37,0.53,437.78
std,6.2,56809.83,50.08,9945.07,13538.19,2.74,4.19,4.58,3.66,0.20,...,11.51,4115.19,25.45,0.58,0.45,1.34,0.46,3.7,0.61,609.45
min,1.0,47703.00,4.68,1034.20,23456.00,37.20,11.20,10.50,5.20,0.15,...,2.80,2385.00,25.62,27.73,48.63,6.68,0.00,0.0,0.0,0.0
25%,6.0,123606.00,6.31,6838.00,31245.00,41.50,16.20,12.30,7.60,0.32,...,4.50,5068.00,34.81,29.52,49.24,7.81,0.00,0.0,0.0,0.0
50%,11.0,149718.00,14.05,15942.50,39876.00,42.80,18.50,16.80,9.80,0.45,...,6.50,6844.00,41.36,29.73,49.46,8.23,1.00,0.0,0.0,0.0
75%,16.0,193104.00,28.24,21962.50,51234.00,44.20,22.10,20.80,12.80,0.58,...,15.80,8106.00,51.79,30.02,49.67,9.40,1.00,4.5,1.0,944.0
max,21.0,257329.00,237.84,29387.20,71432.00,48.10,26.80,25.60,17.20,0.82,...,45.20,22144.00,147.90,30.36,50.65,11.98,1.00,11.0,2.0,1574.0


## 6. Normalización 0-1 para el clustering

Igual que en el socioeconómico: las columnas de `tabla_distritos.csv` vienen en escalas muy distintas (población en cientos de miles, porcentajes de 0 a 100, vulnerabilidad de 0 a 1, tiempo de respuesta en minutos). Para que el clustering compare distancias sin que una columna pese más solo por su escala, pasamos todas las columnas numéricas a 0-1 con min-max: `(valor - mínimo) / (máximo - mínimo)`.

Si una columna tuviera el mismo valor en todos los distritos, el máximo y el mínimo coincidirían y la división daría error (vacío). Esa columna no distingue distritos, así que se deja en 0.

Los valores originales se quedan en `tabla_distritos.csv` y la tabla normalizada se guarda aparte en `tabla_distritos_norm.csv`, que es la que entra al clustering. Las celdas desconocidas de la oferta (unidades, hospitales y camas de los dos Vallecas, y camas de Arganzuela) **se conservan vacías también en la tabla normalizada: no se imputan**. Mientras sigan vacías, esas tres columnas no deben entrar en el clustering; el resto de la tabla está completa y sin huecos.

In [8]:
tabla_guardada = pd.read_csv('tabla_distritos.csv')
cols_norm_tabla = [c for c in tabla_guardada.columns if c not in ('distrito_id', 'distrito')]
tabla_norm = tabla_guardada[['distrito_id', 'distrito']].copy()
for c in cols_norm_tabla:
    minimo, maximo = tabla_guardada[c].min(), tabla_guardada[c].max()
    if maximo > minimo:
        tabla_norm[c] = ((tabla_guardada[c] - minimo) / (maximo - minimo)).round(4)
    else:
        tabla_norm[c] = 0.0   # columna constante: no distingue distritos
tabla_norm.to_csv('tabla_distritos_norm.csv', index=False)
print(tabla_norm.shape)   # (21, 22): mismas columnas, valores entre 0 y 1
vacios = tabla_norm.isna().sum()
print("Vacios por columna (desconocido conservado, no imputado):")
print(vacios[vacios > 0].to_string())
print("Esas columnas no entran en el clustering hasta decidir como tratar el desconocido.")
tabla_norm.describe().round(2)

(21, 22)
Vacios por columna (desconocido conservado, no imputado):
unidades_samur        2
num_hospitales_cat    2
camas_hospital        3
Esas columnas no entran en el clustering hasta decidir como tratar el desconocido.


,distrito_id,poblacion,superficie_km2,densidad_hab_km2,renta_media_hogar,edad_media,pct_mayores_65,pct_extranjeros,tasa_paro,indice_vulnerabilidad,...,parques_m2_habitante,n_activaciones,activaciones_por_1000hab,pct_prioridad_alta,edad_media_paciente,tiempo_respuesta_medio_min,num_bases_samur,unidades_samur,num_hospitales_cat,camas_hospital
count,21.0,21.00,21.00,21.00,21.00,21.00,21.00,21.00,21.00,21.00,...,21.00,21.00,21.00,21.00,21.00,21.00,21.00,19.00,19.00,18.00
mean,11.0,0.52,0.10,0.47,0.38,0.53,0.50,0.40,0.45,0.46,...,0.22,0.24,0.18,0.74,0.42,0.36,0.71,0.22,0.26,0.28
std,6.2,0.27,0.21,0.35,0.28,0.25,0.27,0.30,0.31,0.29,...,0.27,0.21,0.21,0.22,0.22,0.25,0.46,0.34,0.31,0.39
min,1.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,6.0,0.36,0.01,0.20,0.16,0.39,0.32,0.12,0.20,0.25,...,0.04,0.14,0.08,0.68,0.30,0.21,0.00,0.00,0.00,0.00
50%,11.0,0.49,0.04,0.53,0.34,0.51,0.47,0.42,0.38,0.45,...,0.09,0.23,0.13,0.76,0.41,0.29,1.00,0.00,0.00,0.00
75%,16.0,0.69,0.10,0.74,0.58,0.64,0.70,0.68,0.63,0.64,...,0.31,0.29,0.21,0.87,0.51,0.51,1.00,0.41,0.50,0.60
max,21.0,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
